# SynBioCrow 2.3 — BioPKS Post-PKS Diagnostic
Reuses successful PKS-only gate outputs from Drive and isolates downstream DORAnet network generation from pathway post-processing.


In [ ]:
import os,sys,subprocess,shutil,json,zipfile,hashlib
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive",force_remount=False)
REPO=Path("/content/SynBioCrow_postpks_diag")
BIOPKS_ROOT=Path("/content/BioPKS-Pipeline")
BIOPKS_VENV=Path("/content/biopks_py310_postpks")
BIOPKS_PYTHON=BIOPKS_VENV/"bin/python"
GATE=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/biopks_reentry")
OUT=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/biopks_postpks_diagnostic")
OUT.mkdir(parents=True,exist_ok=True)

def run(cmd,*,cwd=None,timeout=1800,check=True):
    print("$"," ".join(map(str,cmd)),flush=True)
    p=subprocess.run(list(map(str,cmd)),cwd=cwd,text=True,
                     stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    print(p.stdout,flush=True)
    if check and p.returncode:
        raise subprocess.CalledProcessError(p.returncode,cmd,output=p.stdout)
    return p

if REPO.exists(): shutil.rmtree(REPO)
run(["git","clone","--depth","1","--branch","develop/2.3",
     "https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)

if BIOPKS_ROOT.exists(): shutil.rmtree(BIOPKS_ROOT)
run(["git","clone","--depth","1","https://github.com/JBEI/BioPKS-Pipeline.git",str(BIOPKS_ROOT)],timeout=600)
run([sys.executable,"-m","pip","install","-q","uv"],timeout=600)
run(["uv","python","install","3.10"],timeout=900)
if BIOPKS_VENV.exists(): shutil.rmtree(BIOPKS_VENV)
run(["uv","venv","--python","3.10",str(BIOPKS_VENV)],timeout=600)
req=BIOPKS_ROOT/"requirements.txt"
patched=BIOPKS_ROOT/"requirements.synbiocrow_colab.txt"
patched.write_text("\n".join(x for x in req.read_text().splitlines() if x.strip().lower()!="umap==0.1.1")+"\n")
run(["uv","pip","install","--python",str(BIOPKS_PYTHON),"-r",str(patched)],timeout=1800)
run(["uv","pip","install","--python",str(BIOPKS_PYTHON),"setuptools<81"],timeout=600)
run(["uv","pip","install","--python",str(BIOPKS_PYTHON),"--no-deps","-e",str(BIOPKS_ROOT)],timeout=900)

panel=json.loads((REPO/"benchmarks/2.3/biopks_reentry_panel.json").read_text())
targets={x["target_id"]:x for x in panel["targets"]}
probe=REPO/"scripts/probe_biopks_post_pks_2_3.py"
summary={"schema":"synbiocrow.biopks_postpks_diagnostic_summary.v1","targets":{}}

for tid in ("pks_naringenin","pks_pinocembrin"):
    saved=GATE/f"{tid}.pks_only.json"
    assert saved.is_file(),f"Missing prior PKS-only checkpoint: {saved}"
    gate=json.loads(saved.read_text())
    routes=(gate.get("response") or {}).get("routes") or []
    assert routes,f"No saved PKS-only routes in {saved}"
    top=max(routes,key=lambda x:float(x.get("score") or -1))
    precursor=top.get("predicted_product_smiles")
    target=targets[tid]["target_smiles"]
    print("\nTARGET",tid,"PKS precursor",precursor,"score",top.get("score"),flush=True)
    summary["targets"][tid]={"pks_precursor":precursor,"pks_score":top.get("score")}

    net_out=OUT/f"{tid}.network.json"
    cmd=[str(BIOPKS_PYTHON),str(probe),"--precursor",precursor,"--target",target,
         "--target-name",tid,"--stage","network","--generations","1","--cores","1","--output",str(net_out)]
    try:
        p=run(cmd,cwd=OUT,timeout=180,check=False)
        summary["targets"][tid]["network_return_code"]=p.returncode
        summary["targets"][tid]["network_result"]=json.loads(net_out.read_text()) if net_out.is_file() else {"status":"NO_REPORT","stdout_tail":p.stdout[-4000:]}
    except subprocess.TimeoutExpired:
        summary["targets"][tid]["network_result"]={"status":"TIMEOUT","timeout_seconds":180}

    if summary["targets"][tid]["network_result"].get("status")=="PASS_NETWORK":
        full_out=OUT/f"{tid}.full.json"
        cmd=[str(BIOPKS_PYTHON),str(probe),"--precursor",precursor,"--target",target,
             "--target-name",tid,"--stage","full","--generations","1","--cores","1","--output",str(full_out)]
        try:
            p=run(cmd,cwd=OUT,timeout=240,check=False)
            summary["targets"][tid]["full_return_code"]=p.returncode
            summary["targets"][tid]["full_result"]=json.loads(full_out.read_text()) if full_out.is_file() else {"status":"NO_REPORT","stdout_tail":p.stdout[-4000:]}
        except subprocess.TimeoutExpired:
            summary["targets"][tid]["full_result"]={"status":"TIMEOUT","timeout_seconds":240}
    else:
        summary["targets"][tid]["full_result"]={"status":"SKIPPED_NETWORK_NOT_BOUNDED"}

report=OUT/"biopks_postpks_diagnostic_summary.json"
report.write_text(json.dumps(summary,indent=2,sort_keys=True)+"\n")
print("\nSUMMARY")
print(json.dumps(summary,indent=2,sort_keys=True))

bundle=OUT/"SynBioCrow_2_3_BIOPKS_POSTPKS_DIAGNOSTIC_RESULTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.glob("*.json")):
        z.write(p,p.name)
    z.write(probe,probe.name)
print("BUNDLE",bundle)
print("SHA256",hashlib.sha256(bundle.read_bytes()).hexdigest())
files.download(str(bundle))
